# Anti-Helmholtz Coils Simulation
This document provides comprehensive analysis and design tools for anti-Helmholtz coil configurations, commonly used in magneto-optical traps (MOTs) and atomic physics experiments.
Includes interactive matplotlib widgets that let you vary key parameters such as coil radius, separation, current, and number of turns to study the resulting magnetic field profiles.

## Brief Theory Introduction

Anti-Helmholtz coils are a pair of circular coils carrying current in opposite directions, separated by a distance typically equal to their radius. Unlike Helmholtz coils which produce a uniform magnetic field at the center, anti-Helmholtz coils create a magnetic field minimum at the center point, with the field increasing linearly in all directions away from the center.

### Magnetic Field from a Single Circular Loop

The magnetic field at a point along the axis of a circular loop of radius $R$ carrying current $I$ at a distance $z$ from the center is:
$$B_z(z) = \frac{\mu_0 I R^2}{2 \left(R^2 + z^2\right)^{3/2}}$$

where $\mu_0 = 4\pi \times 10^{-7}$ T m/A is the magnetic permeability of vacuum.

The radial component is:
$$B_r(z) = \frac{\mu_0 I R z}{2 \left(R^2 + z^2\right)^{3/2}}$$

### Anti-Helmholtz Configuration

For anti-Helmholtz coils, two coils separated by distance $d$ have currents flowing in opposite directions. The field at position $z$ is:
$$B_z^{\mathrm{total}}(z) = B_z^{(1)}\left(z + \frac{d}{2}\right) - B_z^{(2)}\left(z - \frac{d}{2}\right)$$

At the center ($z = 0$), $B_z(0) = 0$, and the field varies linearly near the center:
$$B_z(z) \approx \frac{3 \mu_0 I R^2}{4 \left(R^2 + \left(\frac{d}{2}\right)^2\right)^{5/2}} \cdot z = b' z$$

where $b'$ is the magnetic field gradient (in G/cm).

### Quadrupole Approximation

The 3D magnetic field near the center forms a quadrupole field:
$$\mathbf{B}(x, y, z) \approx b' \begin{pmatrix} x \\ y \\ -2z \end{pmatrix}$$

For ideal anti-Helmholtz with $d = R$, the gradient is:
$$b' = \frac{3 \mu_0 N I}{2 \sqrt{5} R^2}$$

where $N$ is the number of turns in each coil.

### Key Properties

* Field minimum at center: $B(0,0,0) = 0$
* Linear gradient: field increases linearly away from center
* Quadrupole symmetry: field lines point toward center from all directions
* Used in MOTs: provides restoring force for trapped atoms

### Constants and Setup
Here we define physical constants and import necessary libraries.

In [22]:
%matplotlib inline

import math
import numpy as np
from scipy import constants
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as w
from IPython.display import display, clear_output

mu_0 = 4 * math.pi * 1e-7
hbar = constants.hbar
amu = 1.66053906660e-27
mu_B = 9.274009994e-24

cmap_field = plt.cm.RdBu_r

## Single Circular Loop Magnetic Field

Building block of anti-Helmholtz configuration.

In [23]:
def B_single_loop(R, I, z, N=1):
    z = np.asarray(z)
    R2 = R**2
    denominator = (R2 + z**2)**(3/2)
    Bz = (mu_0 * I * N * R2) / (2 * denominator)
    Br = (mu_0 * I * N * R * z) / (2 * denominator)
    B_total = np.sqrt(Bz**2 + Br**2)
    return Bz, Br, B_total

def plot_single_loop(R=0.1, I=10.0, N=1, z_max=0.3):
    z = np.linspace(-z_max, z_max, 500)
    Bz, Br, B_total = B_single_loop(R, I, z, N)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(z * 100, Bz * 1e4, label='Bz', color='blue', lw=2)
    ax1.plot(z * 100, Br * 1e4, label='Br', color='orange', lw=2, ls='--')
    ax1.plot(z * 100, B_total * 1e4, label='|B|', color='red', lw=2)
    ax1.set_xlabel('z (cm)')
    ax1.set_ylabel('B (G)')
    ax1.set_title(f'Single Loop: R={R*100:.1f} cm, I={I:.1f} A, N={N}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0, color='black', lw=0.5)
    
    zoom_range = 0.05
    mask = np.abs(z) <= zoom_range
    ax2.plot(z[mask] * 100, Bz[mask] * 1e4, label='Bz', color='blue', lw=2)
    ax2.plot(z[mask] * 100, B_total[mask] * 1e4, label='|B|', color='red', lw=2)
    ax2.set_xlabel('z (cm)')
    ax2.set_ylabel('B (G)')
    ax2.set_title(f'Zoom: |z| <= {zoom_range*100:.1f} cm')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='black', lw=0.5)
    plt.tight_layout()
    plt.show()

R_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_slider = w.FloatSlider(description='Current (A)', value=10.0, min=1.0, max=50.0, step=0.5)
N_slider = w.IntSlider(description='Turns', value=1, min=1, max=50, step=1)
z_max_slider = w.FloatSlider(description='z range (cm)', value=30.0, min=5.0, max=100.0, step=1.0)

@w.interact(R=R_slider, I=I_slider, N=N_slider, z_max=z_max_slider)
def update_single_loop(R, I, N, z_max):
    plot_single_loop(R/100, I, N, z_max/100)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

## Anti-Helmholtz Coils Configuration

Two coils with opposite currents creating field minimum at center.

In [24]:
def B_anti_helmholtz(R, I, d, z, N=1):
    z = np.asarray(z)
    z1 = z + d/2
    Bz1, Br1, _ = B_single_loop(R, I, z1, N)
    z2 = z - d/2
    Bz2, Br2, _ = B_single_loop(R, -I, z2, N)
    Bz = Bz1 + Bz2
    Br = Br1 + Br2
    B_total = np.sqrt(Bz**2 + Br**2)
    return Bz, Br, B_total

def compute_gradient_at_center(R, I, d, N=1):
    dz = 1e-6
    Bz_pos, _, _ = B_anti_helmholtz(R, I, d, np.array([dz]), N)
    Bz_neg, _, _ = B_anti_helmholtz(R, I, d, np.array([-dz]), N)
    gradient = (Bz_pos[0] - Bz_neg[0]) / (2 * dz)
    return gradient

def plot_anti_helmholtz(R=0.1, I=10.0, d=0.1, N=1, z_max=0.3):
    z = np.linspace(-z_max, z_max, 500)
    Bz, Br, B_total = B_anti_helmholtz(R, I, d, z, N)
    gradient = compute_gradient_at_center(R, I, d, N)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(z * 100, Bz * 1e4, label='Bz', color='blue', lw=2)
    ax1.plot(z * 100, B_total * 1e4, label='|B|', color='red', lw=2)
    ax1.set_xlabel('z (cm)')
    ax1.set_ylabel('B (G)')
    ax1.set_title(f'Anti-Helmholtz: R={R*100:.1f} cm, I={I:.1f} A, N={N}, d/R={d/R:.2f}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0, color='black', lw=0.5)
    
    zoom_range = 0.03
    mask = np.abs(z) <= zoom_range
    ax2.plot(z[mask] * 100, Bz[mask] * 1e4, label='Actual Bz', color='blue', lw=2)
    ax2.plot(z[mask] * 100, gradient * z[mask] * 1e4, 
             label=f'Linear approx: bprime = {abs(gradient)*1e2:.2f} G/cm', 
             color='green', lw=2, ls='--')
    ax2.set_xlabel('z (cm)')
    ax2.set_ylabel('B (G)')
    ax2.set_title(f'Zoom: |z| <= {zoom_range*100:.1f} cm')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='black', lw=0.5)
    plt.tight_layout()
    plt.show()
    return gradient

R_ah_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_ah_slider = w.FloatSlider(description='Current (A)', value=10.0, min=1.0, max=50.0, step=0.5)
d_ah_slider = w.FloatSlider(description='Separation (cm)', value=10.0, min=1.0, max=100.0, step=0.5)
N_ah_slider = w.IntSlider(description='Turns', value=10, min=1, max=100, step=1)
z_max_ah_slider = w.FloatSlider(description='z range (cm)', value=30.0, min=5.0, max=100.0, step=1.0)

gradient_output = w.Output()

@w.interact(R=R_ah_slider, I=I_ah_slider, d=d_ah_slider, N=N_ah_slider, z_max=z_max_ah_slider)
def update_anti_helmholtz(R, I, d, N, z_max):
    gradient = plot_anti_helmholtz(R/100, I, d/100, N, z_max/100)
    with gradient_output:
        clear_output()
        print(f'Gradient: {abs(gradient)*1e2:.4f} G/cm')
        if abs(d - R) < 0.01:
            print('Ideal anti-Helmholtz configuration (d = R)')

display(gradient_output)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

Output()

## Comparison: Helmholtz vs Anti-Helmholtz

Side-by-side comparison of both configurations.

In [25]:
def B_helmholtz(R, I, d, z, N=1):
    z = np.asarray(z)
    z1 = z + d/2
    Bz1, Br1, _ = B_single_loop(R, I, z1, N)
    z2 = z - d/2
    Bz2, Br2, _ = B_single_loop(R, I, z2, N)
    Bz = Bz1 + Bz2
    Br = Br1 + Br2
    B_total = np.sqrt(Bz**2 + Br**2)
    return Bz, Br, B_total

def plot_comparison(R=0.1, I=10.0, d=0.1, N=1, z_max=0.3):
    z = np.linspace(-z_max, z_max, 500)
    Bz_ah, _, _ = B_anti_helmholtz(R, I, d, z, N)
    gradient_ah = compute_gradient_at_center(R, I, d, N)
    Bz_h, _, _ = B_helmholtz(R, I, d, z, N)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(z * 100, Bz_h * 1e4, label='Helmholtz Bz', color='blue', lw=2)
    ax1.plot(z * 100, Bz_ah * 1e4, label='Anti-Helmholtz Bz', color='red', lw=2, ls='--')
    ax1.set_xlabel('z (cm)')
    ax1.set_ylabel('B (G)')
    ax1.set_title(f'Comparison: R={R*100:.0f} cm, d={d*100:.0f} cm')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    zoom_range = 0.05
    mask = np.abs(z) <= zoom_range
    ax2.plot(z[mask] * 100, Bz_h[mask] * 1e4, label='Helmholtz', color='blue', lw=2)
    ax2.plot(z[mask] * 100, Bz_ah[mask] * 1e4, label='Anti-Helmholtz', color='red', lw=2, ls='--')
    ax2.set_xlabel('z (cm)')
    ax2.set_ylabel('B (G)')
    ax2.set_title(f'Center: |z| <= {zoom_range*100:.0f} cm')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    B_center_h = Bz_h[np.argmin(np.abs(z))]
    print(f'Helmholtz: B_center = {B_center_h*1e4:.4f} G (uniform)')
    print(f'Anti-Helmholtz: B_center = {Bz_ah[np.argmin(np.abs(z))]*1e4:.4f} G, Gradient = {abs(gradient_ah)*1e2:.4f} G/cm')

R_comp_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_comp_slider = w.FloatSlider(description='Current (A)', value=10.0, min=1.0, max=50.0, step=0.5)
d_comp_slider = w.FloatSlider(description='Separation (cm)', value=10.0, min=1.0, max=100.0, step=0.5)
N_comp_slider = w.IntSlider(description='Turns', value=10, min=1, max=100, step=1)
z_max_comp_slider = w.FloatSlider(description='z range (cm)', value=30.0, min=5.0, max=100.0, step=1.0)

@w.interact(R=R_comp_slider, I=I_comp_slider, d=d_comp_slider, N=N_comp_slider, z_max=z_max_comp_slider)
def update_comparison(R, I, d, N, z_max):
    plot_comparison(R/100, I, d/100, N, z_max/100)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

## 2D Magnetic Field Visualization

Visualizing the field in the radial-axial plane.

In [ ]:
def B_anti_helmholtz_2d(R, I, d, r, z, N=1):
    R_grid, Z_grid = np.meshgrid(r, z)
    z1 = Z_grid + d/2
    rho1_sq = R_grid**2 + z1**2
    Bz1 = (mu_0 * I * N * R**2) / (2 * (R**2 + rho1_sq)**(3/2))
    Br1 = (mu_0 * I * N * R * R_grid * z1) / (2 * (R**2 + rho1_sq)**(5/2))
    z2 = Z_grid - d/2
    rho2_sq = R_grid**2 + z2**2
    Bz2 = -(mu_0 * I * N * R**2) / (2 * (R**2 + rho2_sq)**(3/2))
    Br2 = -(mu_0 * I * N * R * R_grid * z2) / (2 * (R**2 + rho2_sq)**(5/2))
    Bz = Bz1 + Bz2
    Br = Br1 + Br2
    B_total = np.sqrt(Bz**2 + Br**2)
    return Br, Bz, B_total

def plot_2d_field(R=0.1, I=10.0, d=0.1, N=10, r_max=0.15, z_max=0.15, n_points=100):
    r = np.linspace(-r_max, r_max, n_points)
    z = np.linspace(-z_max, z_max, n_points)
    Br, Bz, B_total = B_anti_helmholtz_2d(R, I, d, r, z, N)
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    R_mesh, Z_mesh = np.meshgrid(r*100, z*100)
    
    contour1 = ax1.contourf(R_mesh, Z_mesh, Bz*1e4, levels=50, cmap=cmap_field)
    ax1.contour(R_mesh, Z_mesh, Bz*1e4, levels=20, colors='black', alpha=0.3)
    ax1.set_xlabel('r (cm)')
    ax1.set_ylabel('z (cm)')
    ax1.set_title('Bz component (G)')
    plt.colorbar(contour1, ax=ax1)
    ax1.axhline(0, color='gray', lw=0.5)
    ax1.axvline(0, color='gray', lw=0.5)
    
    contour2 = ax2.contourf(R_mesh, Z_mesh, B_total*1e4, levels=50, cmap='viridis')
    ax2.contour(R_mesh, Z_mesh, B_total*1e4, levels=20, colors='black', alpha=0.3)
    ax2.set_xlabel('r (cm)')
    ax2.set_ylabel('z (cm)')
    ax2.set_title('|B| magnitude (G)')
    plt.colorbar(contour2, ax=ax2)
    ax2.axhline(0, color='gray', lw=0.5)
    ax2.axvline(0, color='gray', lw=0.5)
    
    r_sparse = np.linspace(-r_max, r_max, 40)
    z_sparse = np.linspace(-z_max, z_max, 40)
    R_sparse, Z_sparse = np.meshgrid(r_sparse, z_sparse)
    Br_sparse, Bz_sparse, _ = B_anti_helmholtz_2d(R, I, d, r_sparse, z_sparse, N)
    ax3.streamplot(R_sparse*100, Z_sparse*100, Br_sparse*1e4, Bz_sparse*1e4, 
                  density=2, color='black', linewidth=0.8)
    ax3.set_xlabel('r (cm)')
    ax3.set_ylabel('z (cm)')
    ax3.set_title('Field lines')
    ax3.axhline(0, color='gray', lw=0.5)
    ax3.axvline(0, color='gray', lw=0.5)
    ax3.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

R_2d_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_2d_slider = w.FloatSlider(description='Current (A)', value=10.0, min=1.0, max=50.0, step=0.5)
d_2d_slider = w.FloatSlider(description='Separation (cm)', value=10.0, min=1.0, max=100.0, step=0.5)
N_2d_slider = w.IntSlider(description='Turns', value=10, min=1, max=100, step=1)
r_max_2d_slider = w.FloatSlider(description='Range (cm)', value=15.0, min=1.0, max=50.0, step=1.0)

@w.interact(R=R_2d_slider, I=I_2d_slider, d=d_2d_slider, N=N_2d_slider, r_max=r_max_2d_slider)
def update_2d_field(R, I, d, N, r_max):
    plot_2d_field(R/100, I, d/100, N, r_max/100, r_max/100)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

## Application to Magneto-Optical Traps (MOTs)

Anti-Helmholtz coils are most commonly used in Magneto-Optical Traps for cooling and trapping neutral atoms. Here we calculate relevant parameters for MOT applications.

In [ ]:
species_data = {
    'Ca-40':   {'mass': 40.078,  'wavelength': 422.67276e-9, 'gamma': 2 * math.pi * 34.6e6},
    'Sr-88':   {'mass': 87.62,   'wavelength': 460.862e-9,    'gamma': 2 * math.pi * 32.0e6},
    'Rb-87':   {'mass': 86.909,  'wavelength': 780.241e-9,    'gamma': 2 * math.pi * 6.065e6},
    'Cs-133':  {'mass': 132.905, 'wavelength': 852.347e-9,    'gamma': 2 * math.pi * 5.234e6},
    'Na-23':   {'mass': 22.990,  'wavelength': 589.158e-9,    'gamma': 2 * math.pi * 9.79e6},
    'Yb-174':  {'mass': 173.045, 'wavelength': 398.911e-9,    'gamma': 2 * math.pi * 29.0e6},
}

def print_mot_params(R, I, d, N, species):
    data = species_data[species]
    mass = data['mass'] * amu
    wavelength = data['wavelength']
    gamma_val = data['gamma']
    
    gradient = compute_gradient_at_center(R, I, d, N)
    k = 2 * math.pi / wavelength
    v_recoil = hbar * k / mass
    # F_max = hbar * k * gamma_val / 2.0
    # a_max = F_max / mass
    T_Doppler = (hbar * gamma_val) / (2 * constants.k)
    
    print('=' * 60)
    print('MOT PARAMETERS')
    print('=' * 60)
    print(f'Configuration: R={R*100:.1f} cm, d={d*100:.1f} cm, I={I:.1f} A, N={N}')
    print(f'Species: {species}')
    print(f'\nField: Gradient = {abs(gradient)*1e2:.4f} G/cm')
    print(f'B at 1cm = {abs(gradient)*0.01*1e4:.2f} G')
    print(f'\nMOT: v_recoil = {v_recoil*100:.4f} cm/s') # a_max = {a_max:.2f} m/s^2')
    print(f'T_Doppler = {T_Doppler*1e6:.2f} microK')
    mass_amu = data['mass']
    print(f'\nSpecies: mass={mass_amu:.1f} amu, lambda={wavelength*1e9:.1f} nm')
    print('=' * 60)

R_mot_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_mot_slider = w.FloatSlider(description='Current (A)', value=5.0, min=0.5, max=50.0, step=0.5)
d_mot_slider = w.FloatSlider(description='Separation (cm)', value=10.0, min=1.0, max=100.0, step=0.5)
N_mot_slider = w.IntSlider(description='Turns', value=10, min=1, max=100, step=1)
species_dropdown = w.Dropdown(description='Species', options=list(species_data.keys()), value='Ca-40')

mot_output = w.Output()

@w.interact(R=R_mot_slider, I=I_mot_slider, d=d_mot_slider, N=N_mot_slider, species=species_dropdown)
def update_mot(R, I, d, N, species):
    with mot_output:
        clear_output()
        print_mot_params(R/100, I, d/100, N, species)

display(mot_output)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

Output()

## Design Tool: Anti-Helmholtz Coils for Target Gradient

This tool helps design anti-Helmholtz coils to achieve a specific magnetic field gradient, which is often the starting point for MOT design.

In [28]:
def design_tool(target_gradient, R, d_ratio, N):
    R_m = R / 100
    d = d_ratio * R_m
    
    denominator = (R_m**2 + (d/2)**2)**(5/2)
    numerator = 3 * mu_0 * N * R_m**2 / (2 * denominator)
    I_req = abs(target_gradient) / numerator
    
    actual_grad = compute_gradient_at_center(R_m, I_req, d, N)
    B_1cm = abs(actual_grad) * 0.01
    
    rho = 1.68e-8
    wire_dia = 1.291e-3
    wire_area = math.pi * (wire_dia/2)**2
    circ = 2 * math.pi * R_m
    wire_len = N * circ * 2
    resistance = rho * wire_len / wire_area
    power = I_req**2 * resistance
    
    print('=' * 60)
    print('ANTI-HELMHOLTZ COIL DESIGN TOOL')
    print('=' * 60)
    print(f'\nTarget Parameters:')
    print(f'  Target gradient = {target_gradient*1e2:.4f} G/cm')
    print(f'\nCoil Geometry:')
    print(f'  Radius R       = {R:.2f} cm')
    print(f'  d/R ratio      = {d_ratio:.3f}')
    print(f'  Separation d   = {d*100:.2f} cm')
    print(f'  Turns N        = {N}')
    print(f'\nRequired Current:')
    print(f'  I = {I_req:.3f} A')
    print(f'\nActual Performance:')
    print(f'  Actual gradient = {abs(actual_grad)*1e2:.4f} G/cm')
    print(f'  B at 1 cm      = {B_1cm*1e4:.2f} G')
    print(f'\nPower Estimate (AWG 16 copper wire):')
    print(f'  Total wire length = {wire_len*100:.2f} m')
    print(f'  Total resistance  = {resistance:.4f} Ohm')
    print(f'  Power dissipation = {power:.2f} W')
    print('=' * 60)

target_grad_slider = w.FloatSlider(description='Gradient (G/cm)', value=15.0, min=5.0, max=50.0, step=1.0)
R_design_slider = w.FloatSlider(description='Radius (cm)', value=15.0, min=2.0, max=30.0, step=0.5)
d_ratio_slider = w.FloatSlider(description='d/R', value=1.0, min=0.5, max=2.0, step=0.05)
N_design_slider = w.IntSlider(description='Turns', value=20, min=1, max=100, step=1)
design_out = w.Output()

@w.interact(target_gradient=target_grad_slider, R=R_design_slider, d_ratio=d_ratio_slider, N=N_design_slider)
def update_design(target_gradient, R, d_ratio, N):
    with design_out:
        clear_output()
        design_tool(target_gradient, R, d_ratio, N)

display(design_out)


interactive(children=(FloatSlider(value=15.0, description='Gradient (G/cm)', max=50.0, min=5.0, step=1.0), Flo…

Output()

In [32]:
def B_loop_axial(R, I, z, N=1):
    denominator = (R**2 + z**2)**(3/2)
    Bz = (mu_0 * I * N * R**2) / (2 * denominator)
    Br = (mu_0 * I * N * R * abs(z)) / (2 * denominator) * np.sign(z)
    return Bz, Br

def compute_gradient_at_center(R, I, d, N=1):
    dz = 1e-6
    Bz_pos, _ = B_loop_axial(R, I, d/2 + dz, N)
    Bz_neg, _ = B_loop_axial(R, I, d/2 - dz, N)
    Bz_pos2, _ = B_loop_axial(R, -I, -d/2 + dz, N)
    Bz_neg2, _ = B_loop_axial(R, -I, -d/2 - dz, N)
    gradient = (Bz_pos + Bz_pos2 - Bz_neg - Bz_neg2) / (2 * dz)
    return gradient

def generate_concentric_radii(r_min, r_max, r_window_start, r_window_end,
                           min_dist_window, min_dist_outside, thickness):
    """
    Generate radii for concentric coils respecting step-function constraints.
    All coils share the same center (0,0).
    """
    radii = []
    current_r = r_min + thickness/2  # Start just outside optical aperture

    while current_r - thickness/2 < r_max:
        # Check which region we're in
        if r_window_start <= current_r <= r_window_end:
            min_spacing = min_dist_window
        else:
            min_spacing = min_dist_outside

        next_r = current_r + thickness + min_spacing

        if next_r - thickness/2 > r_max:
            break

        radii.append(current_r)
        current_r = next_r

    return np.array(radii)

def compute_total_gradient(radii, currents, separation, num_top_bottom):
    """
    Compute total gradient at center from multiple concentric coil pairs.
    Each pair consists of top and bottom coils at same radius.
    """
    total_gradient = 0
    for r, I in zip(radii, currents):
        total_gradient += compute_gradient_at_center(r, I, separation, 1)
    return total_gradient

def plot_cross_section(radii_cm, r_min_cm, r_window_start_cm, r_window_end_cm, r_max_cm, thickness_cm):
    """
    Cross-sectional technical drawing showing concentric coil rings.
    """
    fig, ax = plt.subplots(figsize=(12, 8))

    # Draw optical aperture (minimum radius)
    aperture = plt.Circle((0, 0), r_min_cm, fill=True, color='black', alpha=0.3, label='Optical Aperture')
    ax.add_patch(aperture)

    # Draw window region boundaries
    window_inner = plt.Circle((0, 0), r_window_start_cm, fill=False, color='blue', lw=2, linestyle='--', label='Window Start')
    window_outer = plt.Circle((0, 0), r_window_end_cm, fill=False, color='blue', lw=2, linestyle='--', label='Window End')
    ax.add_patch(window_inner)
    ax.add_patch(window_outer)

    # Draw chamber boundary
    chamber = plt.Circle((0, 0), r_max_cm, fill=False, color='red', lw=2, linestyle=':', label='Chamber End')
    ax.add_patch(chamber)

    # Draw each coil as a ring (two circles for inner and outer radius)
    for i, r in enumerate(radii_cm):
        r_inner = r - thickness_cm/2
        r_outer = r + thickness_cm/2

        # Coil ring
        inner = plt.Circle((0, 0), r_inner, fill=False, color='green', lw=1.5)
        outer = plt.Circle((0, 0), r_outer, fill=False, color='green', lw=1.5)
        ax.add_patch(inner)
        ax.add_patch(outer)

        # Label every few coils
        if i % 2 == 0 or len(radii_cm) <= 5:
            ax.text(0, r + thickness_cm/2 + 1, f'  {r:.1f} cm',
                   va='center', ha='left', fontsize=8)

    # Fill window region manually (two-circle approach)
    window_patch = plt.Circle((0, 0), r_window_end_cm, fill=True, color='lightblue', alpha=0.2)
    ax.add_patch(window_patch)
    inner_patch = plt.Circle((0, 0), r_window_start_cm, fill=True, color='white', alpha=1)
    ax.add_patch(inner_patch)

    # Fill optical aperture
    ax.add_patch(plt.Circle((0, 0), r_min_cm, fill=True, color='black', alpha=0.2))

    # Add radial lines for reference
    for angle in [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi, 5*np.pi/4, 3*np.pi/2, 7*np.pi/4]:
        ax.plot([0, r_max_cm * np.cos(angle)], [0, r_max_cm * np.sin(angle)],
               'gray', lw=0.5, alpha=0.5)

    max_r = r_max_cm + 5
    ax.set_xlim(-max_r, max_r)
    ax.set_ylim(-max_r, max_r)
    ax.set_aspect('equal')
    ax.axhline(0, color='black', lw=0.5)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_xlabel('x (cm)')
    ax.set_ylabel('y (cm)')
    ax.set_title('Cross-Sectional View: Concentric Coils with Step Constraints')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

def optimize_concentric(target_gradient_gcm, max_current, thickness_cm,
                       r_min_cm, r_window_start_cm, r_window_end_cm, r_max_cm,
                       min_dist_window_cm, min_dist_outside_cm, d_ratio=1.0):
    thickness_m = thickness_cm / 100
    r_min_m = r_min_cm / 100
    r_window_start_m = r_window_start_cm / 100
    r_window_end_m = r_window_end_cm / 100
    r_max_m = r_max_cm / 100
    min_dist_window_m = min_dist_window_cm / 100
    min_dist_outside_m = min_dist_outside_cm / 100
    target_gradient_tm = target_gradient_gcm / 100

    # Generate radii for concentric coils
    radii_m = generate_concentric_radii(
        r_min_m, r_max_m, r_window_start_m, r_window_end_m,
        min_dist_window_m, min_dist_outside_m, thickness_m
    )

    if len(radii_m) == 0:
        print("No valid coil configuration with given constraints!")
        return None

    # For anti-Helmholtz: each coil has a pair at +z and -z with opposite currents
    # We'll have num_coils radii, each with top and bottom
    separation_m = d_ratio * np.mean(radii_m)  # Use mean radius for separation

    # Calculate gradient per ampere for each radius
    gradients_per_amp = [compute_gradient_at_center(r, 1.0, separation_m, 1) for r in radii_m]

    # Total gradient per ampere (sum of all coil pairs)
    total_grad_per_amp = sum(abs(g) for g in gradients_per_amp)

    # Required current
    required_current = target_gradient_tm / total_grad_per_amp if total_grad_per_amp > 0 else float('inf')

    if required_current > max_current:
        print("Cannot achieve target gradient with given current limit!")
        return None

    # Calculate actual gradient and currents for each coil
    actual_gradient = total_grad_per_amp * required_current * 100  # Convert to G/cm
    currents = [required_current] * len(radii_m)  # Top coils
    bottom_currents = [-required_current] * len(radii_m)  # Bottom coils

    return {
        'radii': radii_m * 100,
        'currents': currents,
        'separation': separation_m * 100,
        'required_current': required_current,
        'actual_gradient': actual_gradient,
        'num_coils': len(radii_m),
        'total_grad_per_amp': total_grad_per_amp * 100,
        'thickness': thickness_cm
    }

def run_optimizer(target_gradient, max_current, thickness,
                  r_min, r_window_start, r_window_end, r_max,
                  min_dist_window, min_dist_outside, d_ratio):
    result = optimize_concentric(
        target_gradient, max_current, thickness,
        r_min, r_window_start, r_window_end, r_max,
        min_dist_window, min_dist_outside, d_ratio
    )

    if result is None:
        return None

    radii = result['radii']
    separation = result['separation']
    current = result['required_current']
    actual_gradient = result['actual_gradient']
    num_coils = result['num_coils']
    total_grad_per_amp = result['total_grad_per_amp']
    thickness = result['thickness']

    error = abs(actual_gradient - target_gradient) / target_gradient * 100

    print('=' * 60)
    print('OPTIMIZED CONCENTRIC COIL DESIGN')
    print('=' * 60)
    print(f'\nTarget: {target_gradient:.2f} G/cm at max {max_current:.1f} A')
    print(f'\nGeometry Constraints:')
    print(f'  Optical aperture (min radius) = {r_min:.2f} cm')
    print(f'  Window: {r_window_start:.2f} - {r_window_end:.2f} cm')
    print(f'  Chamber (max radius) = {r_max:.2f} cm')
    print(f'  Coil thickness = {thickness:.2f} cm')
    print(f'  Min distance (window) = {min_dist_window:.2f} cm')
    print(f'  Min distance (outside) = {min_dist_outside:.2f} cm')
    print(f'  Separation (d/R) = {d_ratio:.2f} * mean radius')
    print(f'\nResults:')
    print(f'  Number of coil pairs = {num_coils}')
    print(f'  Coil radii = {[f"{r:.1f}" for r in radii]} cm')
    print(f'  Required current = {current:.3f} A')
    print(f'  Achieved gradient = {actual_gradient:.2f} G/cm')
    print(f'  Gradient per ampere = {total_grad_per_amp:.4f} G/cm/A')
    print(f'  Error = {error:.4f}%')
    print('=' * 60)

    plot_cross_section(radii, r_min, r_window_start, r_window_end, r_max, thickness)

    return result

optimization_out = w.Output()

@w.interact(
    target_gradient=w.FloatSlider(value=15.0, min=5.0, max=50.0, step=1.0, description='Gradient (G/cm)'),
    max_current=w.FloatSlider(value=10.0, min=0.1, max=50.0, step=0.5, description='Max current (A)'),
    thickness=w.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Thickness (cm)'),
    r_min=w.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5, description='Optical aperture (cm)'),
    r_window_start=w.FloatSlider(value=5.0, min=0.0, max=20.0, step=0.5, description='Window start (cm)'),
    r_window_end=w.FloatSlider(value=10.0, min=0.0, max=30.0, step=0.5, description='Window end (cm)'),
    r_max=w.FloatSlider(value=20.0, min=1.0, max=50.0, step=0.5, description='Chamber end (cm)'),
    min_dist_window=w.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Min dist (window) (cm)'),
    min_dist_outside=w.FloatSlider(value=0.5, min=0.1, max=3.0, step=0.1, description='Min dist (outside) (cm)'),
    d_ratio=w.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description='d/R')
)
def update_optimization(target_gradient, max_current, thickness,
                       r_min, r_window_start, r_window_end, r_max,
                       min_dist_window, min_dist_outside, d_ratio):
    with optimization_out:
        clear_output()
        run_optimizer(target_gradient, max_current, thickness,
                       r_min, r_window_start, r_window_end, r_max,
                       min_dist_window, min_dist_outside, d_ratio)

display(optimization_out)

interactive(children=(FloatSlider(value=15.0, description='Gradient (G/cm)', max=50.0, min=5.0, step=1.0), Flo…

Output()

## Practical Considerations

### Coil Geometry Notes

1. **d/R = 1 (Ideal anti-Helmholtz)**: This is the most common configuration, providing a good balance between gradient strength and field uniformity near the center.

2. **d/R > 1**: Larger separation produces a weaker gradient but extends the region of linear field. Useful for larger MOTs.

3. **d/R < 1**: Smaller separation produces a stronger gradient but with a smaller linear region. Useful for small, tightly confined traps.

4. **Multiple coil pairs**: For more uniform gradients over larger volumes, multiple anti-Helmholtz coil pairs can be used in series.

### Wire Selection

* **AWG 14**: Thicker wire (1.628 mm), lower resistance, good for high currents
* **AWG 16**: Medium wire (1.291 mm), good balance
* **AWG 18**: Thinner wire (1.024 mm), higher resistance, good for compact designs

### Cooling

Anti-Helmholtz coils can dissipate significant power as heat. Consider:
* **Air cooling**: Sufficient for most small-scale MOTs (< 100 W)
* **Water cooling**: Necessary for high-power applications
* **Pulsed operation**: Can reduce average power requirements

### References

* [NIST Atomic Physics Data](https://www.nist.gov/pml/atomic-physics)
* [Magneto-Optical Trap Review](https://arxiv.org/abs/physics/9803027)

## Summary

This notebook provides comprehensive tools for designing and analyzing anti-Helmholtz coil configurations for atomic physics applications, particularly Magneto-Optical Traps (MOTs).

### Key Features:
* Interactive visualization of magnetic field profiles
* Comparison with Helmholtz coil configurations
* 2D field mapping
* MOT-specific parameter calculations
* Design tools for achieving target gradients
* Power optimization for given constraints

### Next Steps:
* Integrate with specific experimental parameters
* Consider thermal effects and cooling requirements
* Add field compensation coil calculations
* Implement more accurate off-axis field calculations using elliptic integrals